# Deep Learning model for smoothing of PSD noisy curve, estimation of peaks and their frequency ranges.

In [46]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.signal as signal
%matplotlib tk
import matplotlib.pyplot as plt

In [47]:
# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [48]:
# =====================================================================
# 1. DATASET CONVERSION & LOADING ROUTINES
# =====================================================================

def convert_masks_to_target_structures(masks_array, sigma=2.0):
    """
    Converts your 2D mask array [N, 2, n_freqs] into targets for the Keypoint U-Net:
      - Channel 0: Center binary mask -> 1D Gaussian Heatmap Target [N, 1, n_freqs]
      - Channel 1: Interval binary mask -> Continuous Width Targets [N, 2, n_freqs]
    """
    N, _, n_freqs = masks_array.shape
    x_axis = np.arange(n_freqs)
    
    target_heatmaps = np.zeros((N, 1, n_freqs), dtype=np.float32)
    target_widths   = np.zeros((N, 2, n_freqs), dtype=np.float32)  # [Left_Width, Right_Width]
    peak_masks      = np.zeros((N, 1, n_freqs), dtype=np.float32)  # Mask active only at true peak bins

    for i in range(N):
        center_mask   = masks_array[i, 0]   # Channel 0
        interval_mask = masks_array[i, 1]   # Channel 1
        
        # Locate peak center indices (where center_mask == 1.0)
        peak_indices = np.where(center_mask > 0.5)[0]
        
        # 1. Generate 1D Gaussian Heatmap for Peak Head
        for p_idx in peak_indices:
            gaussian = np.exp(-((x_axis - p_idx) ** 2) / (2 * (sigma ** 2)))
            target_heatmaps[i, 0] = np.maximum(target_heatmaps[i, 0], gaussian)
            
        # 2. Extract Left/Right Interval Widths for Width Head
        for p_idx in peak_indices:
            # Step left to measure interval start
            left_bound = p_idx
            while left_bound > 0 and interval_mask[left_bound - 1] > 0.5:
                left_bound -= 1
                
            # Step right to measure interval end
            right_bound = p_idx
            while right_bound < n_freqs - 1 and interval_mask[right_bound + 1] > 0.5:
                right_bound += 1
                
            left_width  = float(p_idx - left_bound)
            right_width = float(right_bound - p_idx)
            
            # Store at peak location
            target_widths[i, 0, p_idx] = left_width   # Left frequency interval (in bins)
            target_widths[i, 1, p_idx] = right_width  # Right frequency interval (in bins)
            peak_masks[i, 0, p_idx]    = 1.0          # Active peak flag for loss computation

    return (
        torch.tensor(target_heatmaps, dtype=torch.float32),
        torch.tensor(target_widths, dtype=torch.float32),
        torch.tensor(peak_masks, dtype=torch.float32)
    )

class ConvertedKeypointPSDDataset(Dataset):
    """
    Dataset class that loads your existing NPZ file format:
    log_noisy_{split}, log_clean_{split}, masks_{split}
    """
    def __init__(self, filepath="psd_dataset_masks.npz", split="train", sigma=2.0):
        assert split in ["train", "val", "test"]
        print(f"Loading split '{split}' from {filepath}...")
        
        data = np.load(filepath, allow_pickle=True)
        self.f = data["f"]
        
        log_noisy_data = data[f"log_noisy_{split}"]
        log_clean_data = data[f"log_clean_{split}"]
        masks_data     = data[f"masks_{split}"]       # Shape: [N, 2, n_freqs]
        
        self.log_noisy = torch.tensor(log_noisy_data, dtype=torch.float32)
        self.log_clean = torch.tensor(log_clean_data, dtype=torch.float32)
        
        # Enforce 3D shapes [N, 1, n_freqs]
        if self.log_noisy.ndim == 2:
            self.log_noisy = self.log_noisy.unsqueeze(1)
        if self.log_clean.ndim == 2:
            self.log_clean = self.log_clean.unsqueeze(1)
            
        # Convert 2D masks into continuous Keypoint Targets
        self.target_heatmap, self.target_width, self.peak_mask = convert_masks_to_target_structures(
            masks_data, sigma=sigma
        )
        print(f" -> Successfully loaded & converted {len(self.log_noisy)} samples.")

    def __len__(self):
        return len(self.log_noisy)

    def __getitem__(self, idx):
        return {
            "noisy_psd":      self.log_noisy[idx],      # Input: [1, n_freqs]
            "clean_psd":      self.log_clean[idx],      # Head 1 Target: [1, n_freqs]
            "target_heatmap": self.target_heatmap[idx], # Head 2 Target: [1, n_freqs]
            "target_width":   self.target_width[idx],   # Head 3 Target: [2, n_freqs]
            "peak_mask":      self.peak_mask[idx]       # Mask for Loss: [1, n_freqs]
        }

In [49]:
# =====================================================================
# 2. MODEL ARCHITECTURE (MULTI-TASK 1D U-NET)
# =====================================================================

class ConvBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.GELU()
        )
    def forward(self, x):
        return self.block(x)

# class ConvBlock1D(nn.Module):
#     def __init__(self, in_c, out_c):
#         super().__init__()
#         self.block = nn.Sequential(
#             nn.Conv1d(in_c, out_c, kernel_size=15, padding=7),
#             nn.BatchNorm1d(out_c),
#             nn.GELU(),
#             nn.Conv1d(out_c, out_c, kernel_size=15, padding=7),
#             nn.BatchNorm1d(out_c),
#             nn.GELU(),
#         )

#     def forward(self, x):
#         return self.block(x)

class MultiTaskUNet1D(nn.Module):
    def __init__(self, in_channels=1, base_filters=32):
        super().__init__()
        # Encoder
        self.enc1 = ConvBlock1D(in_channels, base_filters)
        self.pool1 = nn.MaxPool1d(2)
        self.enc2 = ConvBlock1D(base_filters, base_filters * 2)
        self.pool2 = nn.MaxPool1d(2)
        self.enc3 = ConvBlock1D(base_filters * 2, base_filters * 4)
        
        # Shared Decoder
        self.up2 = nn.ConvTranspose1d(base_filters * 4, base_filters * 2, kernel_size=2, stride=2)
        self.dec2 = ConvBlock1D(base_filters * 4, base_filters * 2)
        
        self.up1 = nn.ConvTranspose1d(base_filters * 2, base_filters, kernel_size=2, stride=2)
        self.dec1 = ConvBlock1D(base_filters * 2, base_filters)
        
        # --- Head 1: Denoising Head (Clean Curve Reconstruction) ---
        self.head_denoise = nn.Sequential(
            nn.Conv1d(base_filters, base_filters // 2, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(base_filters // 2, 1, kernel_size=1)
        )
        
        # --- Head 2: Peak Center Heatmap Head ---
        self.head_heatmap = nn.Sequential(
            nn.Conv1d(base_filters, base_filters // 2, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(base_filters // 2, 1, kernel_size=1)
        )
        
        # --- Head 3: Interval / Bandwidth Width Head ---
        self.head_width = nn.Sequential(
            nn.Conv1d(base_filters, base_filters // 2, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(base_filters // 2, 2, kernel_size=1), # 2 channels: [left_width, right_width]
            nn.ReLU()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        
        d2 = self.up2(e3)
        # Handle odd length padding mismatches from transposed convolution
        if d2.shape[-1] != e2.shape[-1]:
            d2 = F.interpolate(d2, size=e2.shape[-1], mode='linear', align_corners=False)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = self.up1(d2)
        if d1.shape[-1] != e1.shape[-1]:
            d1 = F.interpolate(d1, size=e1.shape[-1], mode='linear', align_corners=False)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        clean_psd     = self.head_denoise(d1)
        heatmap_logits = self.head_heatmap(d1)
        widths         = self.head_width(d1)
        
        return clean_psd, heatmap_logits, widths

In [50]:
# =====================================================================
# 3. LOSS FUNCTIONS & METRICS
# =====================================================================

class ModifiedFocalLoss1D(nn.Module):
    """ CenterNet Focal Loss for keypoint heatmaps """
    def __init__(self, alpha=2.0, beta=4.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, pred_logits, target_heatmap):
        pred = torch.sigmoid(pred_logits)
        pos_mask = target_heatmap.eq(1.0).float()
        neg_mask = target_heatmap.lt(1.0).float()

        neg_weights = torch.pow(1.0 - target_heatmap, self.beta)
        pos_loss = torch.log(pred + 1e-6) * torch.pow(1.0 - pred, self.alpha) * pos_mask
        neg_loss = torch.log(1.0 - pred + 1e-6) * torch.pow(pred, self.alpha) * neg_weights * neg_mask

        num_pos = pos_mask.sum()
        pos_loss = pos_loss.sum()
        neg_loss = neg_loss.sum()

        if num_pos == 0:
            return -neg_loss
        return -(pos_loss + neg_loss) / num_pos


class MultiTaskPSDLoss(nn.Module):
    def __init__(self, w_psd=1.0, w_heatmap=2.0, w_width=1.0):
        super().__init__()
        self.w_psd = w_psd
        self.w_heatmap = w_heatmap
        self.w_width = w_width
        
        self.psd_loss = nn.SmoothL1Loss()
        self.heatmap_loss = ModifiedFocalLoss1D()
        self.width_loss = nn.SmoothL1Loss(reduction='sum')

    def forward(self, pred_psd, pred_logits, pred_widths, batch):
        # 1. Curve Reconstruction Loss
        l_psd = self.psd_loss(pred_psd, batch['clean_psd'])
        
        # 2. Keypoint Heatmap Loss
        l_heatmap = self.heatmap_loss(pred_logits, batch['target_heatmap'])
        
        # 3. Masked Width Loss (Evaluated strictly at ground-truth peak bins)
        mask = batch['peak_mask'].repeat(1, 2, 1) # Expand to 2 width channels
        num_peaks = batch['peak_mask'].sum() + 1e-6
        l_width = self.width_loss(pred_widths * mask, batch['target_width'] * mask) / num_peaks

        total_loss = (self.w_psd * l_psd) + (self.w_heatmap * l_heatmap) + (self.w_width * l_width)
        return total_loss, {"l_psd": l_psd.item(), "l_heatmap": l_heatmap.item(), "l_width": l_width.item()}

class MultiTaskSmoothPSDLoss(nn.Module):
    def __init__(
        self, 
        w_psd=1.0, 
        w_heatmap=2.0, 
        w_width=1.0, 
        lambda_grad=1, 
        lambda_curv=2, 
        lambda_bg=0.05
    ):
        super().__init__()
        self.w_psd = w_psd
        self.w_heatmap = w_heatmap
        self.w_width = w_width
        self.lambda_grad = lambda_grad
        self.lambda_curv = lambda_curv
        self.lambda_bg = lambda_bg
        
        self.psd_loss = nn.SmoothL1Loss()
        self.heatmap_loss = ModifiedFocalLoss1D()
        self.width_loss = nn.SmoothL1Loss(reduction='sum')
        self.mse_loss = nn.MSELoss()

    def forward(self, pred_psd, pred_logits, pred_widths, batch):
        # 1. Base PSD Reconstruction Loss
        l_psd = self.psd_loss(pred_psd, batch['clean_psd'])
        
        # 1b. First Derivative Loss (Gradient Smoothness)
        grad_pred = pred_psd[:, :, 1:] - pred_psd[:, :, :-1]
        grad_true = batch['clean_psd'][:, :, 1:] - batch['clean_psd'][:, :, :-1]
        l_grad = self.mse_loss(grad_pred, grad_true)
        
        # 1c. Second Derivative Loss (Curvature Smoothness)
        curv_pred = pred_psd[:, :, 2:] - 2 * pred_psd[:, :, 1:-1] + pred_psd[:, :, :-2]
        curv_true = batch['clean_psd'][:, :, 2:] - 2 * batch['clean_psd'][:, :, 1:-1] + batch['clean_psd'][:, :, :-2]
        l_curv = self.mse_loss(curv_pred, curv_true)
        
        # Total Smooth Curve Loss
        total_l_psd = l_psd + (self.lambda_grad * l_grad) + (self.lambda_curv * l_curv)

        # 2. Keypoint Heatmap Loss
        l_heatmap = self.heatmap_loss(pred_logits, batch['target_heatmap'])
        
        # 3a. Peak Width Loss (Evaluated strictly at true peak positions)
        mask = batch['peak_mask'].repeat(1, 2, 1) # [B, 2, n_freqs]
        num_peaks = batch['peak_mask'].sum() + 1e-6
        l_width_peak = self.width_loss(pred_widths * mask, batch['target_width'] * mask) / num_peaks

        # 3b. Option 2: Background Width Regularization (Pulls non-peak width predictions toward zero)
        bg_mask = 1.0 - mask
        num_bg = bg_mask.sum() + 1e-6
        l_width_bg = self.mse_loss(pred_widths * bg_mask, torch.zeros_like(pred_widths))
        
        total_l_width = l_width_peak + (self.lambda_bg * l_width_bg)

        # Total Multi-Task Loss
        total_loss = (self.w_psd * total_l_psd) + (self.w_heatmap * l_heatmap) + (self.w_width * total_l_width)
        
        return total_loss, {
            "l_psd": total_l_psd.item(),
            "l_grad": l_grad.item(),
            "l_curv": l_curv.item(),
            "l_heatmap": l_heatmap.item(),
            "l_width_peak": l_width_peak.item(),
            "l_width_bg": l_width_bg.item()
        }

class NormalizedMultiTaskSmoothPSDLoss(nn.Module):
    def __init__(self, ema_alpha=0.99):
        super().__init__()
        self.ema_alpha = ema_alpha
        
        self.psd_loss = nn.SmoothL1Loss()
        self.heatmap_loss = ModifiedFocalLoss1D()
        self.width_loss = nn.SmoothL1Loss(reduction='sum')
        self.mse_loss = nn.MSELoss()

        # Running average registers to scale each of the 6 loss components dynamically
        self.register_buffer('running_l_psd', torch.tensor(1.0))
        self.register_buffer('running_l_grad', torch.tensor(1.0))
        self.register_buffer('running_l_curv', torch.tensor(1.0))
        self.register_buffer('running_l_heatmap', torch.tensor(1.0))
        self.register_buffer('running_l_width_peak', torch.tensor(1.0))
        self.register_buffer('running_l_width_bg', torch.tensor(1.0))

    def forward(self, pred_psd, pred_logits, pred_widths, batch):
        # =====================================================================
        # 1. COMPUTE RAW LOSSES (ALL 6 COMPONENTS)
        # =====================================================================
        
        # Component 0: Base PSD Reconstruction
        l_psd_raw = self.psd_loss(pred_psd, batch['clean_psd'])
        
        # Component 1: First Derivative (Gradient Smoothness)
        grad_pred = pred_psd[:, :, 1:] - pred_psd[:, :, :-1]
        grad_true = batch['clean_psd'][:, :, 1:] - batch['clean_psd'][:, :, :-1]
        l_grad_raw = self.mse_loss(grad_pred, grad_true)
        
        # Component 2: Second Derivative (Curvature Smoothness)
        curv_pred = pred_psd[:, :, 2:] - 2 * pred_psd[:, :, 1:-1] + pred_psd[:, :, :-2]
        curv_true = batch['clean_psd'][:, :, 2:] - 2 * batch['clean_psd'][:, :, 1:-1] + batch['clean_psd'][:, :, :-2]
        l_curv_raw = self.mse_loss(curv_pred, curv_true)
        
        # Component 3: Keypoint Heatmap
        l_heatmap_raw = self.heatmap_loss(pred_logits, batch['target_heatmap'])
        
        # Component 4: Peak Width
        mask = batch['peak_mask'].repeat(1, 2, 1) # [B, 2, n_freqs]
        num_peaks = batch['peak_mask'].sum() + 1e-6
        l_width_peak_raw = self.width_loss(pred_widths * mask, batch['target_width'] * mask) / num_peaks
        
        # Component 5: Background Width Regularization
        bg_mask = 1.0 - mask
        l_width_bg_raw = self.mse_loss(pred_widths * bg_mask, torch.zeros_like(pred_widths))

        # =====================================================================
        # 2. UPDATE RUNNING MEANS (DURING TRAINING ONLY)
        # =====================================================================
        if self.training:
            with torch.no_grad():
                self.running_l_psd        = self.ema_alpha * self.running_l_psd        + (1 - self.ema_alpha) * l_psd_raw.detach()
                self.running_l_grad       = self.ema_alpha * self.running_l_grad       + (1 - self.ema_alpha) * l_grad_raw.detach()
                self.running_l_curv       = self.ema_alpha * self.running_l_curv       + (1 - self.ema_alpha) * l_curv_raw.detach()
                self.running_l_heatmap    = self.ema_alpha * self.running_l_heatmap    + (1 - self.ema_alpha) * l_heatmap_raw.detach()
                self.running_l_width_peak = self.ema_alpha * self.running_l_width_peak + (1 - self.ema_alpha) * l_width_peak_raw.detach()
                self.running_l_width_bg   = self.ema_alpha * self.running_l_width_bg   + (1 - self.ema_alpha) * l_width_bg_raw.detach()

        # =====================================================================
        # 3. NORMALIZE LOSSES TO SAME ORDER OF MAGNITUDE (~1.0)
        # =====================================================================
        l_psd_norm        = l_psd_raw        / (self.running_l_psd        + 1e-6)
        l_grad_norm       = l_grad_raw       / (self.running_l_grad       + 1e-6)
        l_curv_norm       = l_curv_raw       / (self.running_l_curv       + 1e-6)
        l_heatmap_norm    = l_heatmap_raw    / (self.running_l_heatmap    + 1e-6)
        l_width_peak_norm = l_width_peak_raw / (self.running_l_width_peak + 1e-6)
        l_width_bg_norm   = l_width_bg_raw   / (self.running_l_width_bg   + 1e-6)

        total_loss = (
            l_psd_norm + 
            l_grad_norm + 
            l_curv_norm + 
            l_heatmap_norm + 
            l_width_peak_norm + 
            l_width_bg_norm
        )

        # =====================================================================
        # 4. RETURN TOTAL LOSS & DETAILED MONITORING DICTIONARY
        # =====================================================================
        return total_loss, {
            "total_loss": total_loss.item(),
            # Raw values (useful to inspect natural scales)
            "l_psd_raw": l_psd_raw.item(),
            "l_grad_raw": l_grad_raw.item(),
            "l_curv_raw": l_curv_raw.item(),
            "l_heatmap_raw": l_heatmap_raw.item(),
            "l_width_peak_raw": l_width_peak_raw.item(),
            "l_width_bg_raw": l_width_bg_raw.item(),
            # Normalized values (what actually drives training, all ~1.0)
            "l_psd_norm": l_psd_norm.item(),
            "l_grad_norm": l_grad_norm.item(),
            "l_curv_norm": l_curv_norm.item(),
            "l_heatmap_norm": l_heatmap_norm.item(),
            "l_width_peak_norm": l_width_peak_norm.item(),
            "l_width_bg_norm": l_width_bg_norm.item()
        }

class KendallUncertaintySmoothPSDLoss(nn.Module):
    """
    Multi-Task Loss using Kendall Uncertainty Weighting (CVPR 2018).
    
    Automatically learns task variance parameters (log_vars) via gradient descent
    to dynamically balance all 6 loss components to the same order of magnitude.
    
    Components:
      0: Base PSD Reconstruction (Smooth L1)
      1: First Derivative / Gradient Smoothness (MSE)
      2: Second Derivative / Curvature Smoothness (MSE)
      3: Peak Keypoint Center Heatmap (1D Focal Loss)
      4: Peak Interval Width (Masked Smooth L1)
      5: Background Interval Width Regularization (MSE)
    """
    def __init__(self):
        super().__init__()
        # 6 learnable log-variance parameters s_i = log(sigma_i^2)
        # Initialized to zeros (sigma_i = 1.0) so all components start with equal weight
        self.log_vars = nn.Parameter(torch.zeros(6))
        
        # Loss sub-modules
        self.psd_loss = nn.SmoothL1Loss()
        self.heatmap_loss = ModifiedFocalLoss1D()
        self.width_loss = nn.SmoothL1Loss(reduction='sum')
        self.mse_loss = nn.MSELoss()

    def forward(self, pred_psd, pred_logits, pred_widths, batch):
        # =====================================================================
        # 1. COMPUTE RAW UNWEIGHTED LOSSES
        # =====================================================================
        
        # Component 0: Base PSD Reconstruction
        l_psd = self.psd_loss(pred_psd, batch['clean_psd'])
        
        # Component 1: First Derivative (Gradient)
        grad_pred = pred_psd[:, :, 1:] - pred_psd[:, :, :-1]
        grad_true = batch['clean_psd'][:, :, 1:] - batch['clean_psd'][:, :, :-1]
        l_grad = self.mse_loss(grad_pred, grad_true)
        
        # Component 2: Second Derivative (Curvature)
        curv_pred = pred_psd[:, :, 2:] - 2 * pred_psd[:, :, 1:-1] + pred_psd[:, :, :-2]
        curv_true = batch['clean_psd'][:, :, 2:] - 2 * batch['clean_psd'][:, :, 1:-1] + batch['clean_psd'][:, :, :-2]
        l_curv = self.mse_loss(curv_pred, curv_true)
        
        # Component 3: Keypoint Heatmap
        l_heatmap = self.heatmap_loss(pred_logits, batch['target_heatmap'])
        
        # Component 4: Peak Width (Evaluated strictly at active peak positions)
        mask = batch['peak_mask'].repeat(1, 2, 1)  # Expand to [B, 2, n_freqs]
        num_peaks = batch['peak_mask'].sum() + 1e-6
        l_width_peak = self.width_loss(pred_widths * mask, batch['target_width'] * mask) / num_peaks
        
        # Component 5: Background Width Regularization (Pulls non-peak widths to zero)
        bg_mask = 1.0 - mask
        l_width_bg = self.mse_loss(pred_widths * bg_mask, torch.zeros_like(pred_widths))

        # =====================================================================
        # 2. APPLY KENDALL UNCERTAINTY FORMULA TO ALL 6 COMPONENTS
        #    Formula per component: 0.5 * exp(-s_i) * L_i + 0.5 * s_i
        # =====================================================================
        raw_losses = [l_psd, l_grad, l_curv, l_heatmap, l_width_peak, l_width_bg]
        weighted_losses = []
        
        for i, l_raw in enumerate(raw_losses):
            s_i = self.log_vars[i]
            # Precision term: 0.5 * exp(-s_i) * Loss
            # Penalty term:   0.5 * s_i (prevents s_i from blowing up to infinity)
            l_weighted = 0.5 * torch.exp(-s_i) * l_raw + 0.5 * s_i
            weighted_losses.append(l_weighted)
            
        total_loss = sum(weighted_losses)

        # Extract learned standard deviations (sigma_i = exp(0.5 * s_i)) for inspection
        sigmas = torch.exp(0.5 * self.log_vars).detach().cpu().numpy().tolist()

        # =====================================================================
        # 3. RETURN TOTAL LOSS & DETAILED METRICS
        # =====================================================================
        return total_loss, {
            "total_loss": total_loss.item(),
            # Raw unweighted loss values
            "l_psd_raw": l_psd.item(),
            "l_grad_raw": l_grad.item(),
            "l_curv_raw": l_curv.item(),
            "l_heatmap_raw": l_heatmap.item(),
            "l_width_peak_raw": l_width_peak.item(),
            "l_width_bg_raw": l_width_bg.item(),
            # Weighted loss values (what actually drives backpropagation)
            "l_psd_weighted": weighted_losses[0].item(),
            "l_grad_weighted": weighted_losses[1].item(),
            "l_curv_weighted": weighted_losses[2].item(),
            "l_heatmap_weighted": weighted_losses[3].item(),
            "l_width_peak_weighted": weighted_losses[4].item(),
            "l_width_bg_weighted": weighted_losses[5].item(),
            # Learned uncertainty scale parameters (sigmas)
            "sigma_psd": sigmas[0],
            "sigma_grad": sigmas[1],
            "sigma_curv": sigmas[2],
            "sigma_heatmap": sigmas[3],
            "sigma_width_peak": sigmas[4],
            "sigma_width_bg": sigmas[5],
        }

In [51]:
# =====================================================================
# 4. EVALUATION ROUTINE (1D Non-Maximum Suppression + Metrics)
# =====================================================================

def extract_peaks_1d(heatmap_prob, threshold=0.3, kernel_size=5):
    """ Extracts discrete peak centers via 1D Local Maxima Pooling """
    pad = (kernel_size - 1) // 2
    hmax = F.max_pool1d(heatmap_prob, kernel_size=kernel_size, stride=1, padding=pad)
    keep = (heatmap_prob == hmax) & (heatmap_prob > threshold)
    
    indices = torch.nonzero(keep.squeeze()).squeeze(-1)
    if indices.ndim == 0 and indices.numel() > 0:
        indices = indices.unsqueeze(0)
    return indices.cpu().numpy()

def evaluate_keypoint_model(model, test_loader, criterion, device, peak_thresh=0.35):
    model.eval()
    test_loss = 0.0
    total_samples = 0
    
    reconstruction_maes = []
    width_errors = []

    with torch.no_grad():
        for batch in test_loader:
            noisy = batch['noisy_psd'].to(device)
            clean = batch['clean_psd'].to(device)
            batch_gpu = {k: v.to(device) for k, v in batch.items()}
            
            pred_psd, pred_logits, pred_widths = model(noisy)
            
            # Loss Calculation
            loss, _ = criterion(pred_psd, pred_logits, pred_widths, batch_gpu)
            test_loss += loss.item() * noisy.size(0)
            total_samples += noisy.size(0)
            
            # Curve MAE
            mae = torch.mean(torch.abs(pred_psd - clean)).item()
            reconstruction_maes.append(mae)
            
            # Width MAE at active peaks
            mask = batch_gpu['peak_mask'].repeat(1, 2, 1)
            num_peaks = mask.sum().item()
            if num_peaks > 0:
                w_err = (torch.abs(pred_widths - batch_gpu['target_width']) * mask).sum().item() / num_peaks
                width_errors.append(w_err)

    test_loss /= total_samples
    mean_mae = np.mean(reconstruction_maes)
    mean_w_err = np.mean(width_errors) if len(width_errors) > 0 else 0.0

    print("\n" + "=" * 55)
    print("      RUNNING EVALUATION ON KEYPOINT TEST SET      ")
    print("=" * 55)
    print(f" * Test Composite Loss       : {test_loss:.6f}")
    print(f" * Reconstruction Curve MAE  : {mean_mae:.4f}")
    print(f" * Mean Interval Width MAE   : {mean_w_err:.4f} frequency bins")
    print("=" * 55 + "\n")

    return {
        "test_loss": test_loss,
        "recon_mae": mean_mae,
        "width_mae": mean_w_err
    }

In [ ]:
# =====================================================================
# 5. MAIN EXECUTION PIPELINE
# =====================================================================

def train_keypoint_psd_model(
    dataset_path="psd_dataset_masks.npz", 
    save_dir="checkpoints_keypoints",
    epochs=15, 
    batch_size=32, 
    lr=1e-3
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training Keypoint Multi-Task PSD Model on device: {device}")

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "best_keypoint_unet.pth")

    # 1. Load Datasets & Dataloaders
    train_ds = ConvertedKeypointPSDDataset(filepath=dataset_path, split="train")
    val_ds   = ConvertedKeypointPSDDataset(filepath=dataset_path, split="val")
    test_ds  = ConvertedKeypointPSDDataset(filepath=dataset_path, split="test")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    # 2. Instantiate Model, Loss, Optimizer
    model = MultiTaskUNet1D(in_channels=1, base_filters=32).to(device)
    # criterion = MultiTaskPSDLoss(w_psd=1.0, w_heatmap=2.0, w_width=1.0)
    # criterion = MultiTaskSmoothPSDLoss(w_psd=1.0, w_heatmap=0.2, w_width=0.1)
    # criterion = NormalizedMultiTaskSmoothPSDLoss()
    criterion = KendallUncertaintySmoothPSDLoss()
    #optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # NOTE: for Kendall criterion
    optimizer = torch.optim.AdamW([
        {'params': model.parameters(), 'lr': 1e-3},
        {'params': criterion.parameters(), 'lr': 1e-2}  # 10x higher LR for uncertainty parameters
    ], weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

    best_val_loss = float("inf")

    # 3. Training Loop
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for batch in train_loader:
            noisy_psd = batch['noisy_psd'].to(device)
            batch_gpu = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            pred_psd, pred_logits, pred_widths = model(noisy_psd)
            
            loss, _ = criterion(pred_psd, pred_logits, pred_widths, batch_gpu)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * noisy_psd.size(0)

        train_loss /= len(train_ds)

        # Validation Step
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                noisy_psd = batch['noisy_psd'].to(device)
                batch_gpu = {k: v.to(device) for k, v in batch.items()}

                pred_psd, pred_logits, pred_widths = model(noisy_psd)
                loss, _ = criterion(pred_psd, pred_logits, pred_widths, batch_gpu)
                val_loss += loss.item() * noisy_psd.size(0)

        val_loss /= len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch [{epoch:02d}/{epochs:02d}] | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
        print(_)
        # Save Checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
            }
            torch.save(checkpoint, save_path)

    print(f"\nTraining Complete! Best Val Loss: {best_val_loss:.6f} | Saved to '{save_path}'")

    # 4. Load Best Model and Run Evaluation
    checkpoint = torch.load(save_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    
    test_metrics = evaluate_keypoint_model(model, test_loader, criterion, device)
    
    return model, test_metrics

In [53]:
if __name__ == "__main__":
    # Ensure dataset filename matches your generated dataset path
    DATASET_FILENAME = "psd_dataset_masks.npz" 
    
    trained_model, test_metrics = train_keypoint_psd_model(
        dataset_path=DATASET_FILENAME,
        epochs=30,
        batch_size=32,
        lr=1e-3
    )

Training Keypoint Multi-Task PSD Model on device: cuda
Loading split 'train' from psd_dataset_masks.npz...
 -> Successfully loaded & converted 20000 samples.
Loading split 'val' from psd_dataset_masks.npz...
 -> Successfully loaded & converted 2500 samples.
Loading split 'test' from psd_dataset_masks.npz...
 -> Successfully loaded & converted 2500 samples.
Epoch [01/30] | Train Loss: 9.179161 | Val Loss: 7.742759
{'total_loss': 5.036780834197998, 'l_psd_raw': 0.011978003196418285, 'l_grad_raw': 0.00286676874384284, 'l_curv_raw': 0.00350429629907012, 'l_heatmap_raw': 0.46997666358947754, 'l_width_peak_raw': 7.39175271987915, 'l_width_bg_raw': 2.193483591079712, 'l_psd_weighted': 0.005989001598209143, 'l_grad_weighted': 0.00143338437192142, 'l_curv_weighted': 0.00175214814953506, 'l_heatmap_weighted': 0.23498833179473877, 'l_width_peak_weighted': 3.695876359939575, 'l_width_bg_weighted': 1.096741795539856, 'sigma_psd': 1.0, 'sigma_grad': 1.0, 'sigma_curv': 1.0, 'sigma_heatmap': 1.0, 'sig

In [54]:
# =====================================================================
# 1. PEAK & BANDWIDTH DECODER (1D NMS)
# =====================================================================

def decode_keypoint_predictions(pred_logits, pred_widths, f_grid, center_thresh=0.35, kernel_size=5):
    """
    Decodes keypoint heatmap logits and predicted width vectors into physical frequencies.
    
    Returns:
        peaks_info: List of dicts containing f0, f_left, f_right, and heatmap_score
    """
    # 1. Apply Sigmoid to heatmap logits
    heatmap_prob = torch.sigmoid(pred_logits)  # Shape: [1, 1, n_freqs]
    
    # 2. Local Maxima Pooling (1D Non-Maximum Suppression)
    pad = (kernel_size - 1) // 2
    hmax = F.max_pool1d(heatmap_prob, kernel_size=kernel_size, stride=1, padding=pad)
    keep = (heatmap_prob == hmax) & (heatmap_prob >= center_thresh)
    
    peak_indices = torch.nonzero(keep.squeeze()).squeeze(-1)
    if peak_indices.ndim == 0 and peak_indices.numel() > 0:
        peak_indices = peak_indices.unsqueeze(0)
        
    peak_indices = peak_indices.cpu().numpy()
    
    # Frequency bin step size (df in Hz)
    df = f_grid[1] - f_grid[0] if len(f_grid) > 1 else 1.0
    
    peaks_info = []
    pred_widths_np = pred_widths.squeeze().cpu().numpy()  # [2, n_freqs]
    probs_np = heatmap_prob.squeeze().cpu().numpy()        # [n_freqs]
    
    for idx in peak_indices:
        f0 = f_grid[idx]
        score = probs_np[idx]
        
        # Read predicted left/right bin widths from Head 3
        left_bins = pred_widths_np[0, idx]
        right_bins = pred_widths_np[1, idx]
        
        # Convert bin offsets into Hz bounds
        f_left = max(f_grid[0], f0 - (left_bins * df))
        f_right = min(f_grid[-1], f0 + (right_bins * df))
        
        peaks_info.append({
            "idx": idx,
            "f0": f0,
            "f_left": f_left,
            "f_right": f_right,
            "score": score
        })
        
    return peaks_info, probs_np


# =====================================================================
# 2. KEYPOINT RESULT VISUALIZER
# =====================================================================

def plot_keypoint_psd_inference(f_grid, log_noisy, pred_clean, peaks_info, heatmap_prob):
    """
    Plots the empirical noisy log PSD, the network's clean curve reconstruction, 
    the continuous 1D peak heatmap, and the detected peak center/interval regions.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2.5, 1]})

    # --- Top Subplot: PSD Curves and Peak Bounds ---
    ax1.plot(f_grid, log_noisy, label="Noisy Empirical Welch PSD", color="gray", alpha=0.5, linestyle="--")
    ax1.plot(f_grid, pred_clean, label="Keypoint U-Net Denoised PSD", color="crimson", linewidth=2.0)

    # Draw detected peak centers and interval bounds
    for i, p in enumerate(peaks_info):
        # Vertical line for Peak Center (f0)
        ax1.axvline(x=p["f0"], color="blue", linestyle=":", linewidth=1.5, 
                    label="Detected Peak Center" if i == 0 else "")
        
        # Shaded frequency interval [f_left, f_right]
        ax1.axvspan(p["f_left"], p["f_right"], color="royalblue", alpha=0.2, 
                    label="Predicted Frequency Interval" if i == 0 else "")
        
        # Annotation text
        peak_y = pred_clean[p["idx"]]
        ax1.annotate(
            f"Peak {i+1}: {p['f0']:.2f} Hz\n[{p['f_left']:.1f} - {p['f_right']:.1f} Hz]",
            xy=(p["f0"], peak_y),
            xytext=(p["f0"], peak_y + 0.6),
            ha='center',
            arrowprops=dict(arrowstyle="->", color="black", lw=1),
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5)
        )

    ax1.set_ylabel("Log Power Spectral Density")
    ax1.set_title("Keypoint Multi-Task U-Net: PSD Denoising & Peak Interval Detection")
    ax1.grid(True, linestyle=":", alpha=0.6)
    ax1.legend(loc="upper right")

    # --- Bottom Subplot: Predicted Center Heatmap ---
    ax2.plot(f_grid, heatmap_prob, color="darkorange", linewidth=1.8, label="Predicted Center Heatmap Head")
    ax2.axhline(y=0.35, color="black", linestyle="--", alpha=0.6, label="Threshold (0.35)")
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Probability")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend(loc="upper right")

    plt.tight_layout()
    plt.show()


# =====================================================================
# 3. MODEL LOAD & INFERENCE EXECUTION SCRIPT
# =====================================================================

def load_keypoint_model(checkpoint_path="checkpoints_keypoints/best_keypoint_unet.pth", base_filters=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint file '{checkpoint_path}' not found!")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    model = MultiTaskUNet1D(in_channels=1, base_filters=base_filters).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Loaded Keypoint UNet from '{checkpoint_path}'")
    return model, device


if __name__ == "__main__":
    CHECKPOINT_PATH = "checkpoints_keypoints/best_keypoint_unet.pth"
    
    # --- Settings ---
    USE_SIMULATED = False   # Toggle False for real EEG recording
    TARGET_N_FREQS = 250   # Matches training resolution
    TARGET_F_MAX = 50.0    # Frequency ceiling (50 Hz)

    if USE_SIMULATED:
        print("\n--- Running Keypoint Inference on Synthetic Signal ---")
        fs = 250.0
        t = np.arange(0, 10, 1/fs)
        
        # Synthetic EEG signal: 1/f background + 10 Hz Alpha peak + 22 Hz Beta peak + Noise
        clean_signal = np.sin(2 * np.pi * 10 * t) + 0.6 * np.sin(2 * np.pi * 22 * t)
        noise = np.random.normal(0, 1.5, size=len(t))
        y = clean_signal + noise

    else:
        print("\n--- Running Keypoint Inference on Real EEG Recording File ---")
        file = r"c:\Users\holcman\Documents\GitHub\EEG-labellisation-app---Spectrogram\anesthesia_database\rec_20240321_085300.npy"
        fs = 128
        y = np.load(file)
        y = y[2100 * fs : 2250 * fs]  # Extract 150 second slice
        t = np.arange(len(y)) / fs

    # --- Compute Empirical PSD via Welch's Method ---
    nperseg = int(8 * fs)
    f_emp, psd_emp = signal.welch(y, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)

    # Restrict to [0.1, 50 Hz]
    freq_mask = (f_emp >= 0.1) & (f_emp <= TARGET_F_MAX)
    f_emp = f_emp[freq_mask]
    psd_emp = psd_emp[freq_mask]

    # Resample onto model grid (n_freqs = 250)
    f_grid = np.linspace(0.1, TARGET_F_MAX, TARGET_N_FREQS)
    log_psd_emp = np.log(psd_emp + 1e-12)
    resampled_log_psd = np.interp(f_grid, f_emp, log_psd_emp)

    # Format input tensor [1, 1, n_freqs]
    input_tensor = torch.tensor(resampled_log_psd, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    # --- RUN INFERENCE ---
    model, device = load_keypoint_model(CHECKPOINT_PATH, base_filters=32)
    input_tensor = input_tensor.to(device)

    with torch.no_grad():
        pred_clean_tensor, pred_logits, pred_widths = model(input_tensor)

    # Extract NumPy predictions
    pred_clean_arr = pred_clean_tensor.squeeze().cpu().numpy()
    
    # Decode Heatmaps and Widths into peak information
    peaks_info, heatmap_prob = decode_keypoint_predictions(
        pred_logits, pred_widths, f_grid, center_thresh=0.35
    )

    # Print Summary Logs
    print(f"\n--- Model Output Summary ---")
    print(f"Detected Peak Count: {len(peaks_info)}")
    for i, p in enumerate(peaks_info):
        print(f" Peak {i+1}: Center = {p['f0']:.2f} Hz | Interval = [{p['f_left']:.2f} Hz, {p['f_right']:.2f} Hz] | Heatmap Score = {p['score']:.3f}")

    # --- VISUALIZE RESULTS ---
    plot_keypoint_psd_inference(
        f_grid=f_grid,
        log_noisy=resampled_log_psd,
        pred_clean=pred_clean_arr,
        peaks_info=peaks_info,
        heatmap_prob=heatmap_prob
    )


--- Running Keypoint Inference on Real EEG Recording File ---
Loaded Keypoint UNet from 'checkpoints_keypoints/best_keypoint_unet.pth'

--- Model Output Summary ---
Detected Peak Count: 2
 Peak 1: Center = 0.70 Hz | Interval = [0.33 Hz, 1.27 Hz] | Heatmap Score = 0.868
 Peak 2: Center = 13.13 Hz | Interval = [10.78 Hz, 14.23 Hz] | Heatmap Score = 0.628


In [55]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# =====================================================================
# INFERENCE & RAW WIDTH VISUALIZATION
# =====================================================================

# 1. Forward pass through your model
model.eval()
with torch.no_grad():
    pred_clean_tensor, pred_logits, pred_widths_tensor = model(input_tensor)

# 2. Extract NumPy arrays for visualization
pred_clean = pred_clean_tensor.squeeze().cpu().numpy()               # [n_freqs]
heatmap_prob = torch.sigmoid(pred_logits).squeeze().cpu().numpy()     # [n_freqs]

# Extract Raw Width Channels from Head 3 (Shape: [2, n_freqs])
raw_widths = pred_widths_tensor.squeeze().cpu().numpy()
df = f_grid[1] - f_grid[0]  # Frequency bin resolution in Hz

# Convert raw width predictions from "number of bins" to "Hz"
raw_left_width_hz  = raw_widths[0, :] * df   # Channel 0: Left Width (Hz)
raw_right_width_hz = raw_widths[1, :] * df   # Channel 1: Right Width (Hz)


# =====================================================================
# 3. PLOT ALL 3 HEADS (PSD, HEATMAP, RAW WIDTHS)
# =====================================================================

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1, figsize=(11, 8), sharex=True, gridspec_kw={'height_ratios': [2.5, 1, 1]}
)

# --- Subplot 1: PSD Curves ---
ax1.plot(f_grid, resampled_log_psd, label="Noisy Welch Input", color="gray", alpha=0.5, linestyle="--")
ax1.plot(f_grid, pred_clean, label="Denoised PSD (Head 1)", color="crimson", linewidth=2.0)
ax1.set_ylabel("Log PSD")
ax1.set_title("Multi-Task 1D U-Net: Raw Output Visualization Across Frequency Spectrum")
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.legend(loc="upper right")

# --- Subplot 2: Peak Center Heatmap ---
ax2.plot(f_grid, heatmap_prob, color="darkorange", linewidth=1.8, label="Peak Heatmap Prob (Head 2)")
ax2.axhline(y=0.35, color="black", linestyle="--", alpha=0.6, label="Detection Threshold (0.35)")
ax2.set_ylabel("Probability")
ax2.set_ylim(-0.05, 1.05)
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(loc="upper right")

# --- Subplot 3: Raw Predicted Widths (Head 3) ---
ax3.plot(f_grid, raw_left_width_hz, color="teal", linewidth=1.8, label="Raw Predicted Left Width Δf_L (Hz)")
ax3.plot(f_grid, raw_right_width_hz, color="purple", linewidth=1.8, linestyle="-.", label="Raw Predicted Right Width Δf_R (Hz)")
ax3.set_xlabel("Frequency (Hz)")
ax3.set_ylabel("Width (Hz)")
ax3.grid(True, linestyle=":", alpha=0.6)
ax3.legend(loc="upper right")

plt.tight_layout()
plt.show()